## Generate more high-quality sentiment labels with an LLM

Few-shot sentiment classification using Gemini, based on 600+ human labeled examples.

In [ ]:
from google.colab import ai, drive
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')
cleaned_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/cleaned_forum_posts.parquet')

## Config & Resume State

In [ ]:
from pathlib import Path
from datetime import datetime
import re

TARGET_LABELS  = 10000
SAVE_PATH      = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'
SAVE_EVERY     = 50
N_PER_CLASS    = 5   # 5 per class = 15 few-shot examples total

# Posts that must not be labeled (already have a human label)
human_ids = set(human_labeled_sentiment['id'])
unlabeled = cleaned_sentiment[~cleaned_sentiment['id'].isin(human_ids)]

# Resume: pick up any LLM labels already saved
if Path(SAVE_PATH).exists():
    llm_labeled = pd.read_parquet(SAVE_PATH)
    llm_ids     = set(llm_labeled['id'])
    print(f"Resuming: {len(llm_labeled)} LLM labels already saved.")
else:
    llm_labeled = pd.DataFrame()
    llm_ids     = set()

# Exclude already-labeled posts and shuffle so we don't walk companies in order
to_label  = (unlabeled[~unlabeled['id'].isin(llm_ids)]
             .sample(frac=1, random_state=42)
             .reset_index(drop=True))
remaining = max(0, TARGET_LABELS - len(llm_labeled))

print(f"Companies in pool : {to_label['ticker'].nunique()} / {cleaned_sentiment['ticker'].nunique()}")
print(f"Posts in queue    : {len(to_label):,}")
print(f"Still needed      : {remaining:,}")

Resuming: 10000 LLM labels already saved.
Companies in pool : 162 / 162
Posts in queue    : 521,816
Still needed      : 0


## Few-shot Helpers

In [ ]:
LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive"}


def select_examples(labeled_df, ticker, forum, n_per_class=N_PER_CLASS, seed=42):
    """
    Select 5 examples per class (15 total).
    Hard constraint : same company (ticker) examples are always used first.
    Soft preference : same forum fills remaining slots, then global pool.
    """
    rng = np.random.default_rng(seed)

    parts = []
    for cls in [0, 1, 2]:
        cls_pool = labeled_df[labeled_df['sentiment'] == cls]

        # Hard: same company
        company = cls_pool[cls_pool['ticker'] == ticker]
        n_take  = min(len(company), n_per_class)
        chunk   = [company.sample(n=n_take, random_state=int(rng.integers(1_000_000)))] if n_take else []
        needed  = n_per_class - n_take

        # Soft: same forum (excluding same company)
        if needed:
            forum_others = cls_pool[(cls_pool['forum'] == forum) & (cls_pool['ticker'] != ticker)]
            n_take = min(len(forum_others), needed)
            if n_take:
                chunk.append(forum_others.sample(n=n_take, random_state=int(rng.integers(1_000_000))))
            needed -= n_take

        # Fallback: anything else
        if needed:
            rest   = cls_pool[(cls_pool['ticker'] != ticker) & (cls_pool['forum'] != forum)]
            n_take = min(len(rest), needed)
            if n_take:
                chunk.append(rest.sample(n=n_take, random_state=int(rng.integers(1_000_000))))

        if chunk:
            parts.append(pd.concat(chunk))

    if not parts:
        return pd.DataFrame()

    return (pd.concat(parts)
              .sample(frac=1, random_state=int(rng.integers(1_000_000)))
              .reset_index(drop=True))


def build_prompt(row, examples_df):
    lines = [
        "Classify the sentiment of Finnish-language financial forum posts toward the mentioned company.",
        "Reply with a single digit only: 0 (negative), 1 (neutral), 2 (positive).",
        "",
        "Examples:",
    ]
    for _, ex in examples_df.iterrows():
        lines += [
            f"Company: {ex['company_name']}",
            f"Post: {str(ex['message'])[:600]}",
            f"Label: {int(ex['sentiment'])}",
            "",
        ]
    lines += [
        "Now label this post:",
        f"Company: {row['company_name']}",
        f"Post: {str(row['message'])[:600]}",
        "Label:",
    ]
    return "\n".join(lines)


def parse_label(response):
    s = str(response).strip()
    if s in ('0', '1', '2'):
        return int(s)
    match = re.search(r'[012]', s)
    return int(match.group()) if match else None

## Labeling Loop

In [ ]:
results = llm_labeled.copy()
batch   = []
skipped = 0

print(f"Starting — {len(results)} existing + {remaining} to go (target {TARGET_LABELS}).\n")

for i, (_, row) in enumerate(to_label.iterrows()):
    if len(results) + len(batch) >= TARGET_LABELS:
        break

    examples = select_examples(human_labeled_sentiment, row['ticker'], row['forum'], seed=i)
    if examples.empty:
        skipped += 1
        continue

    prompt = build_prompt(row, examples)

    try:
        response = ai.generate_text(prompt, model_name='google/gemini-2.5-flash')
        label    = parse_label(response)
    except Exception as e:
        print(f"  API error at index {i}: {e}")
        continue

    if label is None:
        skipped += 1
        continue

    entry                = row.to_dict()
    entry['sentiment']   = label
    entry['labeled_at']  = datetime.now().isoformat()
    entry['label_source'] = 'llm'
    batch.append(entry)

    if len(batch) >= SAVE_EVERY:
        results = pd.concat([results, pd.DataFrame(batch)], ignore_index=True)
        results.to_parquet(SAVE_PATH, index=False)
        print(f"  [{len(results)}/{TARGET_LABELS}] saved — {skipped} skipped so far")
        batch = []

# Final flush
if batch:
    results = pd.concat([results, pd.DataFrame(batch)], ignore_index=True)
    results.to_parquet(SAVE_PATH, index=False)

print(f"\nDone. {len(results):,} LLM labels saved to {SAVE_PATH}. Skipped {skipped}.")

Starting — 2500 existing + 7500 to go (target 10000).

  [2550/10000] saved — 0 skipped so far
  [2600/10000] saved — 0 skipped so far
  [2650/10000] saved — 0 skipped so far
  [2700/10000] saved — 0 skipped so far
  [2750/10000] saved — 0 skipped so far
  [2800/10000] saved — 0 skipped so far
  [2850/10000] saved — 0 skipped so far
  [2900/10000] saved — 0 skipped so far
  [2950/10000] saved — 0 skipped so far
  [3000/10000] saved — 0 skipped so far
  [3050/10000] saved — 0 skipped so far
  [3100/10000] saved — 0 skipped so far
  [3150/10000] saved — 0 skipped so far
  [3200/10000] saved — 0 skipped so far
  [3250/10000] saved — 0 skipped so far
  [3300/10000] saved — 0 skipped so far
  [3350/10000] saved — 0 skipped so far
  [3400/10000] saved — 0 skipped so far
  [3450/10000] saved — 0 skipped so far
  [3500/10000] saved — 0 skipped so far
  [3550/10000] saved — 0 skipped so far
  [3600/10000] saved — 0 skipped so far
  [3650/10000] saved — 0 skipped so far
  [3700/10000] saved — 0 

## Results Summary

In [ ]:
llm_results = pd.read_parquet(SAVE_PATH)

In [ ]:
print(f"Total LLM labels: {len(llm_results):,}")
print("\nSentiment distribution:")
print(llm_results['sentiment'].value_counts().sort_index().rename(LABEL_NAMES))
print("\nBy forum:")
print(llm_results.groupby('forum')['sentiment'].value_counts().unstack(fill_value=0).rename(columns=LABEL_NAMES))

Total LLM labels: 10,000

Sentiment distribution:
sentiment
negative    3591
neutral     2782
positive    3627
Name: count, dtype: int64

By forum:
sentiment      negative  neutral  positive
forum                                     
Inderes             656      819       937
Kauppalehti        2823     1881      2597
Sijoitustieto       112       82        93
